In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "sofie"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find the database
db_files = list(DATA_DIR.glob("*.db"))

db_files

[WindowsPath('c:/Users/leena/OneDrive/Desktop/ev-microgrid-twin/data/sofie/sofie_data.db')]

In [2]:
DB_PATH = db_files[0]

conn = sqlite3.connect(
    f"file:{DB_PATH.as_posix()}?mode=ro",
    uri=True
)

conn.execute("PRAGMA query_only=ON")

# Inspect table columns
pd.read_sql_query(
    "PRAGMA table_info(scn_values)",
    conn
)

,cid,name,type,notnull,dflt_value,pk
0,0,id,BIGINT,0,None,0
1,1,timestamp,DATETIME,0,None,0
2,2,SCN_L1,FLOAT,0,None,0
3,3,SCN_L2,FLOAT,0,None,0
4,4,SCN_L3,FLOAT,0,None,0
5,5,max_L1,BIGINT,0,None,0
6,6,max_L2,BIGINT,0,None,0
7,7,max_L3,BIGINT,0,None,0


In [3]:
pd.read_sql_query(
    "PRAGMA table_info(logging)",
    conn
)

,cid,name,type,notnull,dflt_value,pk
0,0,id,BIGINT,0,None,0
1,1,timestamp,DATETIME,0,None,0
2,2,U_1,BIGINT,0,None,0
3,3,U_2,BIGINT,0,None,0
4,4,U_3,BIGINT,0,None,0
5,5,I_1,BIGINT,0,None,0
6,6,I_2,BIGINT,0,None,0
7,7,I_3,BIGINT,0,None,0
8,8,I_n,BIGINT,0,None,0
9,9,P_sum,BIGINT,0,None,0


In [4]:
query = """
SELECT
    timestamp,
    bat_soc,
    bat_temp,
    bat_cur,
    bat_vol
FROM logging
ORDER BY timestamp
"""

battery = pd.read_sql_query(query, conn)

battery.head()

,timestamp,bat_soc,bat_temp,bat_cur,bat_vol
0,2025-12-01 00:00:02.000000,45,123,1606,4928
1,2025-12-01 00:00:07.000000,45,123,1634,4928
2,2025-12-01 00:00:12.000000,45,123,1634,4928
3,2025-12-01 00:00:17.000000,45,123,1634,4928
4,2025-12-01 00:00:22.000000,45,123,1661,4928


In [5]:
battery.info()

battery.describe()

<class 'pandas.DataFrame'>
RangeIndex: 3607431 entries, 0 to 3607430
Data columns (total 5 columns):
 #   Column     Dtype
---  ------     -----
 0   timestamp  str  
 1   bat_soc    int64
 2   bat_temp   int64
 3   bat_cur    int64
 4   bat_vol    int64
dtypes: int64(4), str(1)
memory usage: 137.6 MB


,bat_soc,bat_temp,bat_cur,bat_vol
count,3.607431e+06,3.607431e+06,3.607431e+06,3.607431e+06
mean,6.949876e+01,1.783373e+02,-7.698340e+02,4.983839e+03
std,2.569693e+01,6.571707e+01,4.471394e+04,8.017437e+01
min,2.500000e+01,7.500000e+01,-3.370520e+05,4.640000e+03
25%,4.200000e+01,1.270000e+02,-1.780000e+02,4.925000e+03
50%,7.300000e+01,1.610000e+02,2.046000e+03,4.981000e+03
75%,9.700000e+01,2.120000e+02,3.007000e+03,5.018000e+03
max,1.000000e+02,3.950000e+02,4.207140e+05,5.379000e+03


In [6]:
battery["timestamp"] = pd.to_datetime(battery["timestamp"])

battery["bat_temp_c"] = battery["bat_temp"] / 10
battery["bat_current_a"] = battery["bat_cur"] / 1000

# Keep the original values for reference
battery["bat_soc_pct"] = battery["bat_soc"]

# Check data quality
print("Missing values:")
print(battery.isnull().sum())

print("\nDuplicate timestamps:")
print(battery["timestamp"].duplicated().sum())

print("\nConverted statistics:")
print(
    battery[[
        "bat_soc_pct",
        "bat_temp_c",
        "bat_current_a",
        "bat_vol"
    ]].describe()
)

print("\nExtreme current readings:")
print(
    battery.loc[
        battery["bat_current_a"].abs() > 100,
        ["timestamp", "bat_current_a"]
    ].head(10)
)

Missing values:
timestamp        0
bat_soc          0
bat_temp         0
bat_cur          0
bat_vol          0
bat_temp_c       0
bat_current_a    0
bat_soc_pct      0
dtype: int64

Duplicate timestamps:
3

Converted statistics:
        bat_soc_pct    bat_temp_c  bat_current_a       bat_vol
count  3.607431e+06  3.607431e+06   3.607431e+06  3.607431e+06
mean   6.949876e+01  1.783373e+01  -7.698340e-01  4.983839e+03
std    2.569693e+01  6.571707e+00   4.471394e+01  8.017437e+01
min    2.500000e+01  7.500000e+00  -3.370520e+02  4.640000e+03
25%    4.200000e+01  1.270000e+01  -1.780000e-01  4.925000e+03
50%    7.300000e+01  1.610000e+01   2.046000e+00  4.981000e+03
75%    9.700000e+01  2.120000e+01   3.007000e+00  5.018000e+03
max    1.000000e+02  3.950000e+01   4.207140e+02  5.379000e+03

Extreme current readings:
               timestamp  bat_current_a
5564 2025-12-01 07:43:42        100.430
5565 2025-12-01 07:43:47        137.784
5566 2025-12-01 07:43:52        139.212
5567 2025-12-01 0

In [7]:
battery["bat_voltage_v"] = battery["bat_vol"] / 100

print("Voltage statistics:")
print(battery["bat_voltage_v"].describe())

print("\nDuplicate timestamp records:")
print(
    battery[battery["timestamp"].duplicated(keep=False)]
    .sort_values("timestamp")
)

Voltage statistics:
count    3.607431e+06
mean     4.983839e+01
std      8.017437e-01
min      4.640000e+01
25%      4.925000e+01
50%      4.981000e+01
75%      5.018000e+01
max      5.379000e+01
Name: bat_voltage_v, dtype: float64

Duplicate timestamp records:
                 timestamp  bat_soc  bat_temp  bat_cur  bat_vol  bat_temp_c  \
369862 2025-12-22 09:52:02       38       128    -6852     4940        12.8   
369863 2025-12-22 09:52:02       38       128    -7512     4939        12.8   
369864 2025-12-22 09:52:07       38       128    -7319     4939        12.8   
369865 2025-12-22 09:52:07       38       128    -7484     4939        12.8   
369866 2025-12-22 09:52:12       38       128    -7402     4940        12.8   
369867 2025-12-22 09:52:12       38       128    -7704     4939        12.8   

        bat_current_a  bat_soc_pct  bat_voltage_v  
369862         -6.852           38          49.40  
369863         -7.512           38          49.39  
369864         -7.319       

In [8]:
# Select useful converted measurements
battery_values = [
    "bat_soc_pct",
    "bat_temp_c",
    "bat_current_a",
    "bat_voltage_v"
]

# Average records sharing the same timestamp
battery_by_time = battery.groupby("timestamp")[battery_values].mean()

# Aggregate into 15-minute intervals
battery_15min = battery_by_time.resample("15min").agg({
    "bat_soc_pct": "mean",
    "bat_temp_c": "mean",
    "bat_current_a": "mean",
    "bat_voltage_v": "mean"
})

# Count available timestamp readings in each interval
battery_15min["reading_count"] = (
    battery_by_time["bat_soc_pct"].resample("15min").count()
)

battery_15min.head()

,bat_soc_pct,bat_temp_c,bat_current_a,bat_voltage_v,reading_count
timestamp,,,,,
2025-12-01 00:00:00,45.0,12.241667,1.623911,49.280000,180
2025-12-01 00:15:00,45.0,12.200000,1.633389,49.279167,180
2025-12-01 00:30:00,45.0,12.158889,1.623283,49.273278,180
2025-12-01 00:45:00,45.0,12.100000,1.627622,49.271222,180
2025-12-01 01:00:00,45.0,12.100000,1.633078,49.270056,180


In [9]:
print("Total intervals:", len(battery_15min))
print("\nMissing values:")
print(battery_15min.isnull().sum())

print("\nReading count statistics:")
print(battery_15min["reading_count"].describe())

Total intervals: 20256

Missing values:
bat_soc_pct      211
bat_temp_c       211
bat_current_a    211
bat_voltage_v    211
reading_count      0
dtype: int64

Reading count statistics:
count    20256.000000
mean       178.091825
std         18.349017
min          0.000000
25%        180.000000
50%        180.000000
75%        180.000000
max        181.000000
Name: reading_count, dtype: float64


In [10]:
# Identify intervals with available battery readings
battery_15min["data_available"] = (
    battery_15min["reading_count"] > 0
)

# Identify intervals with approximately complete readings
battery_15min["data_complete"] = (
    battery_15min["reading_count"] >= 171
)

# Export
battery_output = battery_15min.reset_index()

output_path = OUTPUT_DIR / "battery_15min.csv"
battery_output.to_csv(output_path, index=False)

print("Exported to:", output_path)
print("\nComplete intervals:", battery_15min["data_complete"].sum())
print("Missing intervals:", (~battery_15min["data_available"]).sum())
print("Partial intervals:", (
    battery_15min["data_available"] &
    ~battery_15min["data_complete"]
).sum())

Exported to: c:\Users\leena\OneDrive\Desktop\ev-microgrid-twin\data\processed\battery_15min.csv

Complete intervals: 20029
Missing intervals: 211
Partial intervals: 16
